# encoder-decoder-symmetric — faded example 1: Tiny U-Net Block: Two-Stage Symmetric Module

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `encoder-decoder-symmetric`. The last cell reports your progress on the `CNN: Encoder-decoder symmetric layout` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Encoder-decoder symmetric layout` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`encoder-decoder-symmetric`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "encoder-decoder-symmetric"
DD_SUBTOPIC = "CNN: Encoder-decoder symmetric layout"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The encoder-decoder pattern extends naturally to any number of downsampling stages — the critical rule is simply that each pool or strided-conv in the encoder has a corresponding upsample or transposed-conv in the decoder with the same factor. The channel count reverses symmetrically: if the encoder goes C → 2C → 4C, the decoder goes 4C → 2C → C.

## Faded exercise 1

Implement `make_symmetric_ae(in_channels, base_channels)` that builds an `nn.Module` with a two-stage symmetric encoder-decoder. The encoder is `Conv2d(in_channels, base_channels, k=3, p=1) → ReLU → MaxPool2d(2) → Conv2d(base_channels, base_channels*2, k=3, p=1) → ReLU → MaxPool2d(2)`. The decoder mirrors this exactly, ending without a final ReLU. Return the module.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t
import torch.nn as nn

def make_symmetric_ae(in_channels: int, base_channels: int) -> nn.Module:
    class SymAE(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = nn.Sequential(
                nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(base_channels, base_channels * 2, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
            )
            self.decoder = nn.Sequential(
                nn.Upsample(scale_factor=2),
                nn.Conv2d(base_channels * 2, base_channels, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Upsample(scale_factor=2),
                nn.Conv2d(base_channels, in_channels, kernel_size=3, padding=1),
            )

        def forward(self, x):
            return self.decoder(self.encoder(x))

    return SymAE()


def _test():
    import torch as t
    model = make_symmetric_ae(in_channels=3, base_channels=8)
    model.eval()
    for h in [16, 32]:
        x = t.randn(2, 3, h, h)
        out = model(x)
        assert out.shape == x.shape, f"Shape mismatch at H={h}: {out.shape}"
    # channel count check
    enc_last = list(model.encoder.children())[-1]
    assert isinstance(enc_last, t.nn.MaxPool2d)


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

def make_symmetric_ae(in_channels: int, base_channels: int) -> nn.Module:
    class SymAE(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = nn.Sequential(
                nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(base_channels, base_channels * 2, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
            )
            self.decoder = nn.Sequential(
                nn.Upsample(scale_factor=2),
                nn.Conv2d(base_channels * 2, base_channels, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Upsample(scale_factor=2),
                nn.Conv2d(base_channels, in_channels, kernel_size=3, padding=1),
            )

        def forward(self, x):
            return self.decoder(self.encoder(x))

    return SymAE()
```
</details>